In [16]:
import os
import sys

# Detect if executing inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Arkadyg27/TimeSeriesProject.git"
    PROJECT_DIR = "/content/TimeSeriesProject"

    os.chdir('/content')
    if not os.path.exists(PROJECT_DIR):
        !git clone {REPO_URL}
    else:
        os.chdir(PROJECT_DIR)
        !git pull

    os.chdir(PROJECT_DIR)
    print("Google Colab detected. Working directory set to:", os.getcwd())
else:
    print("Running locally. Working directory set to:", os.getcwd())


Already up to date.
Google Colab detected. Working directory set to: /content/TimeSeriesProject


In [17]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install mlflow rasterio pymannkendall
else:
    print("Local environment detected. Make sure dependencies are installed.")


In [18]:
import sys
IN_COLAB = 'google.colab' in sys.modules

# 1. Native Colab Authentication
if IN_COLAB:
    try:
        from google.colab import auth
        auth.authenticate_user()
    except Exception as e:
        print('Colab auth notice:', e)

import ee

# Team Earth Engine Projects
PROJECT_IDS = ['889258893131', 'timeseriesproject-503021']

initialized = False
for proj in PROJECT_IDS:
    try:
        ee.Initialize(project=proj)
        print(f'Earth Engine initialized successfully using project: "{proj}"')
        initialized = True
        break
    except Exception:
        continue

if not initialized:
    try:
        ee.Initialize()
        print('Earth Engine initialized using account default project!')
    except Exception:
        print('Prompting interactive Earth Engine authentication...')
        ee.Authenticate()
        ee.Initialize()


Earth Engine initialized successfully using project: "889258893131"


In [19]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules

# Sync full and partial download checkpoints from Google Drive to preserve download progress
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_cache_dir = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject'

    if os.path.exists(drive_cache_dir):
        !cp -n "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/"*.parquet /content/TimeSeriesProject/ 2>/dev/null || true
        print('Synced dataset and checkpoint caches from Google Drive!')
    else:
        !mkdir -p "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Synced dataset and checkpoint caches from Google Drive!


In [20]:
%env MLFLOW_ALLOW_FILE_STORE=true
import os, sys
import mlflow
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Define project path in Google Drive
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject'
    MLRUNS_DIR = os.path.join(DRIVE_PROJECT_PATH, 'mlruns')
    os.makedirs(MLRUNS_DIR, exist_ok=True)

    # Direct MLflow to log persistently to Google Drive
    os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
    os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{DRIVE_PROJECT_PATH}/mlflow.db"
    mlflow.set_tracking_uri(f"sqlite:///{DRIVE_PROJECT_PATH}/mlflow.db")
    print(f"MLflow tracking initialized! Runs will be saved to: {MLRUNS_DIR}")
else:
    print("Local environment: MLflow tracking locally.")


env: MLFLOW_ALLOW_FILE_STORE=true
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MLflow tracking initialized! Runs will be saved to: /content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/mlruns


In [22]:
# ========================================================
# SMART PREPROCESSING: Skip if files already exist in Drive
# ========================================================
import os, shutil, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    DRIVE_PREPROCESS = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/Preprocess'
    LOCAL_PREPROCESS = 'Preprocess'

    # Check if preprocessed files already exist in Google Drive
    check_file = os.path.join(DRIVE_PREPROCESS, 'Altamira_NDVI_CenteredMatrix.parquet')

    if os.path.exists(check_file):
        print("Found preprocessed data in Google Drive! Skipping preprocessing...")
        # Instantly link Google Drive files to local workspace
        os.makedirs(LOCAL_PREPROCESS, exist_ok=True)
        for file_name in os.listdir(DRIVE_PREPROCESS):
            src = os.path.join(DRIVE_PREPROCESS, file_name)
            dst = os.path.join(LOCAL_PREPROCESS, file_name)
            if not os.path.exists(dst):
                os.symlink(src, dst)
        DRIVE_TIFF = os.path.join(DRIVE_PROJECT_PATH, "Tiff")
        LOCAL_TIFF = "Tiff"
        if os.path.exists(DRIVE_TIFF):
            import shutil
            if os.path.exists(LOCAL_TIFF) and not os.path.islink(LOCAL_TIFF):
                shutil.rmtree(LOCAL_TIFF)
            if not os.path.exists(LOCAL_TIFF):
                os.symlink(DRIVE_TIFF, LOCAL_TIFF)
        print("All preprocessed data linked! Ready to train.")
    else:
        print("First-time run: Preprocessing data...")
        !python run_preprocessing.py

        # Save a backup to Google Drive so you never run it again
        shutil.copytree(LOCAL_PREPROCESS, DRIVE_PREPROCESS, dirs_exist_ok=True)
        print("Preprocessed files backed up to Google Drive for future sessions!")
else:
    print("Local environment: Running preprocessing...")
    !python run_preprocessing.py


Found preprocessed data in Google Drive! Skipping preprocessing...
All preprocessed data linked! Ready to train.


In [ ]:
!python run_baseline_all.py


In [23]:
!python Altamira_Modis_repro.py

Starting experiments for Altamira (NDVI)...
Loading cached dataset from data_Altamira_ndvi.parquet...
Raw data loaded. Shape: (64449, 276)

--- Running Isolation Forest (leak_free=False) ---
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Altamira_NDVI_IsolationForest_leaky_est_20.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Altamira_NDVI_IsolationForest_leaky_est_40.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Altamira_NDVI_IsolationForest_leaky_est_60.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Altamira_NDVI_IsolationForest_leaky_est_80.tif

--- Running Isolation Forest (leak_free=True) ---
Skipping: Model already trained! Found cached result at Tiff/leak_free/IsolationForest/Altamira_NDVI_IsolationForest_leakfree_est_20.tif
Skipping: Model already trained! Found cached result at Tiff/leak_free/IsolationForest/Altamira_NDVI_

In [24]:
!python Brumadinho_Sentinel_repro.py

Starting experiments for Brumadinho...

==================== Band: NDWI ====================
Loading cached dataset from data_Brumadinho_ndwi.parquet...
Raw data loaded. Shape: (99540, 157)

--- Running Isolation Forest (leak_free=False) ---
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Brumadinho_NDWI_IsolationForest_leaky_est_20.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Brumadinho_NDWI_IsolationForest_leaky_est_40.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Brumadinho_NDWI_IsolationForest_leaky_est_60.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Brumadinho_NDWI_IsolationForest_leaky_est_80.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Brumadinho_NDWI_IsolationForest_leaky_est_100.tif

--- Running Isolation Forest (leak_free=True) ---
Skipping: Model already trained! Found cached 

In [25]:
!python Mariana_Landsat_repro.py

Starting experiments for Mariana...

==================== Band: GVMI ====================
Loading cached dataset from data_Mariana_gvmi.parquet...
Raw data loaded. Shape: (52863, 122)

--- Running Isolation Forest (leak_free=False) ---
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Mariana_GVMI_IsolationForest_leaky_est_20.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Mariana_GVMI_IsolationForest_leaky_est_40.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Mariana_GVMI_IsolationForest_leaky_est_60.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Mariana_GVMI_IsolationForest_leaky_est_80.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Mariana_GVMI_IsolationForest_leaky_est_100.tif

--- Running Isolation Forest (leak_free=True) ---
Skipping: Model already trained! Found cached result at Tiff/leak_f

In [ ]:
!python train_deep.py

==================== Deep Learning: Altamira NDVI (LSTM Autoencoder) ====================
Loading precomputed centered matrix...
Extracting Time-Aware Features (Velocity, Acceleration, Rolling Stats)...
Calculating velocity...
Calculating acceleration...
Calculating rolling stats (window=3)...
Stacking features into tensor...
--- USING DEVICE: cpu ---
Training up to 10 epochs...
Epoch [1/10], Loss: 0.025778, Time: 451.84s
Epoch [2/10], Loss: 0.022394, Time: 303.83s
Epoch [3/10], Loss: 0.022392, Time: 314.25s
Epoch [4/10], Loss: 0.022390, Time: 305.66s
Epoch [5/10], Loss: 0.022388, Time: 304.18s
Epoch [6/10], Loss: 0.022384, Time: 304.05s
Epoch [7/10], Loss: 0.022380, Time: 306.01s
Epoch [8/10], Loss: 0.022380, Time: 303.01s
Epoch [9/10], Loss: 0.022368, Time: 304.50s


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Save full parquet datasets and partial download checkpoints to Google Drive
    !mkdir -p "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject"
    !cp -u *.parquet "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/" 2>/dev/null || true
    print('Backed up parquet datasets and checkpoints to Google Drive.')

    # Save output results (Tiff & mlruns)
    !mkdir -p "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject"
    !cp -r Tiff/ "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/"
    !cp -r mlruns/ "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/"
    print("Backed up results to /content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/")
else:
    print("Results and checkpoints are stored locally.")
